# **Libraries**

In [ ]:
# --------------------------------------------------------
#  Import Libraries
# --------------------------------------------------------
import base64, json, time, msal, requests, pyodbc
from typing import Dict, List, Optional
import notebookutils.credentials as cred

# **Azure Key Vault**

In [ ]:
# --------------------------------------------------------
# Define Azure Key Vault
# --------------------------------------------------------
key_vault_name = "hcm-prod-kv"
key_vault_url = f"https://{key_vault_name}.vault.azure.net/"

# --------------------------------------------------------
# Get Azure Key Vault Service Principal Credentials
# --------------------------------------------------------
client_name = "HCM-SPN-Fabric"
tenant_id = cred.getSecret(key_vault_url,"FabricTenantId")
client_id = cred.getSecret(key_vault_url,"FabricClientId")
client_secret = cred.getSecret(key_vault_url,"FabricClientSecret")

# **Functions**

## **Access token function**

In [ ]:
# --------------------------------------------------------
#  Get access token function
# --------------------------------------------------------
def get_access_token(tenant_id, client_id, client_secret):
    """
    Retrieve a Fabric API access token using a service principal.
    Returns the token string, or raises an exception with a clear message.
    """

    # token_provider_uri = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    token_provider_uri = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"
    return token_provider_uri.strip()

    headers = {'Content-Type': 'application/x-www-form-urlencoded'}

    body = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'resource': 'https://api.fabric.microsoft.com',
        'scope': 'https://api.fabric.microsoft.com/.default'
    }

    try:
        response = requests.post(token_provider_uri, data=body, headers=headers)
        response.raise_for_status()
    except requests.exceptions.HTTPError as http_err:
        # Fabric loves giving vague 401s, so let’s make it obvious
        raise RuntimeError(
            f"Failed to retrieve access token. "
            f"HTTP error: {http_err}. "
            f"Check tenant ID, client ID, secret, and SP permissions."
        )
    except Exception as ex:
        raise RuntimeError(
            f"Unexpected error while retrieving access token: {ex}"
        )

    token = response.json().get("access_token")
    if not token:
        raise RuntimeError("Token request succeeded but no access_token was returned.")

    print("Access token successfully retrieved.")
    return token

# --------------------------------------------------------
#  Get capacity id function
# --------------------------------------------------------
def get_capacity_id(access_token: str, workspace_id: str) -> str:
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()

    data = response.json()

    # capacity ID shows up as: data["capacityId"]
    return data.get("capacityId")

## **Creation functions**

In [ ]:
# --------------------------------------------------------
#  Workspace creation function
# --------------------------------------------------------
def create_workspace(access_token: str, workspace_name: str, capacity_id: str = None) -> None:
    """
    Create a Microsoft Fabric Workspace.

    :param access_token: OAuth2 bearer token from Entra ID.
    :param workspace_name: Display name for the Workspace.
    :param capacity_id: Optional Fabric capacity GUID for assignment.
    """

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": workspace_name
    }

    # Capacity assignment is optional
    if capacity_id:
        body["capacityId"] = capacity_id

    endpoint = "https://api.fabric.microsoft.com/v1/workspaces"

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Workspace '{workspace_name}' created successfully.")

# --------------------------------------------------------
#  Lakehouse creation function
# --------------------------------------------------------
def create_lakehouse(
        access_token: str
      , lakehouse_name: str
      , enable_schemas: bool = False
) -> None:
    """
    Create a Microsoft Fabric Lakehouse in the current workspace.

    :param access_token: OAuth2 bearer token from Entra ID.
    :param lakehouse_name: Display name for the Lakehouse.
    :param enable_schemas: When True, provisions a schema-enabled Lakehouse.
    """

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": lakehouse_name
    }
    # ---------- schema-enabled switch ----------
    if enable_schemas:
        # Preview flag – only True is allowed
        body["creationPayload"] = {"enableSchemas": True}

    # Use the dedicated Lakehouse collection endpoint
    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses"

    response = requests.post(endpoint, headers=headers, json=body)
    # 201 = created, 202 = long-running provisioning
    response.raise_for_status()

    print(f"Lakehouse '{lakehouse_name}' created by: {client_name}")

# --------------------------------------------------------
#  Notebook creation function
# --------------------------------------------------------
def create_notebook(access_token: str, notebook_name: str) -> None:
    """
    Create an empty Fabric Notebook with the given name.
    """
    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/notebooks"

    requests.post(
        endpoint,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json"
        },
        json={"displayName": notebook_name}
    )

    print(f"Notebook '{notebook_name}' created by: {client_name}")

# --------------------------------------------------------
#  Warehouse creation function
# --------------------------------------------------------
def create_warehouse(access_token: str, warehouse_name: str) -> None:
    """
    Create a Fabric Warehouse with the given name.
    """
    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses"

    requests.post(
        endpoint,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json"
        },
        json={"displayName": warehouse_name}
    )

    print(f"Warehouse '{warehouse_name}' created by: {client_name}")

# --------------------------------------------------------
#  SQL Server creation function
# --------------------------------------------------------

def create_sql_server(access_token: str, sql_server_name: str) -> None:
    """
    Create a Fabric SQL database with the given name via service principal.
    """
    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"

    response = requests.post(
        endpoint,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json"
        },
        json={
            "displayName": sql_server_name,
            "type": "SQLDatabase"
        }
    )
    response.raise_for_status()  # Raises error if not 2xx

    print(f"SQL database '{sql_server_name}' created successfully.")

# --------------------------------------------------------    
#  Variable library creation function
# --------------------------------------------------------
def create_variable_library(access_token: str, library_name: str) -> None:
    """
    Create a Microsoft Fabric Variable Library in the current workspace.

    :param access_token: OAuth2 bearer token from Entra ID.
    :param library_name: Display name for the Variable Library.
    :param description: Optional description for the library.
    """
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": library_name
    }

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/VariableLibraries"

    response = requests.post(endpoint, headers=headers, json=body)
    # 201 = created, 202 = provisioning
    response.raise_for_status()

    print(f"Variable Library '{library_name}' created successfully.")

# --------------------------------------------------------
#  Pipeline creation function
# --------------------------------------------------------
def create_pipeline(access_token: str, pipeline_name: str) -> None:
    """
    Create a Fabric Data Pipeline with the given name.
    """
    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/dataPipelines"

    requests.post(
        endpoint,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json"
        },
        json={"displayName": pipeline_name}
    )

    print(f"Pipeline '{pipeline_name}' created by: {client_name}")

# --------------------------------------------------------    
#  Dataflow creation function
# --------------------------------------------------------
def create_dataflow_gen2(access_token: str, dataflow_name: str) -> None:
    """
    Create a Microsoft Fabric Dataflow (Gen2) in the current workspace.

    :param access_token: OAuth2 bearer token from Entra ID.
    :param dataflow_name: Display name for the Dataflow.
    """
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": dataflow_name
    }

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/dataflows"

    response = requests.post(endpoint, headers=headers, json=body)
    # 201 = created, 202 = long-running provisioning
    response.raise_for_status()

    print(f"Dataflow Gen2 '{dataflow_name}' created by: {client_name}")

# --------------------------------------------------------
#  Eventhouse creation function
# --------------------------------------------------------
def create_eventhouse(access_token: str, eventhouse_name: str) -> None:
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": eventhouse_name
    }

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses"

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Eventhouse '{eventhouse_name}' created by: {client_name}")

# --------------------------------------------------------
#  Eventstream creation function
# --------------------------------------------------------
def create_eventstream(access_token: str, eventstream_name: str) -> None:
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": eventstream_name
    }

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventstreams"

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Eventstream '{eventstream_name}' created by: {client_name}")


# --------------------------------------------------------
#  Ontology creation function
# --------------------------------------------------------
def create_ontology(access_token: str, ontology_name: str) -> None:
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": ontology_name
    }

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/ontologies"

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Ontology '{ontology_name}' created by: {client_name}")


# --------------------------------------------------------
#  Reflex (Activator) creation function
# --------------------------------------------------------
def create_reflex(access_token: str, reflex_name: str) -> None:
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": reflex_name
    }

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/reflexes"

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Reflex '{reflex_name}' created by: {client_name}")

# --------------------------------------------------------
#  Mirrored Database creation function
# --------------------------------------------------------
def create_mirrored_database(
        access_token: str,
        mirrored_database_name: str,
        source_type: str,
        connection_string: str
) -> None:
    """
    Create a Microsoft Fabric Mirrored Database in the current workspace.

    :param access_token: OAuth2 access token
    :param name: Display name for the Mirrored Database
    :param source_type: Source system type ("SqlDatabase", "SqlManagedInstance", "SqlServer", "AzurePostgreSql", "CosmosDb")
    :param connection_string: Connection string for the mirrored source
    """

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/mirroredDatabases"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": mirrored_database_name,
        "source": {
            "type": source_type,
            "properties": {
                "connectionString": connection_string
            }
        }
    }

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    result = response.json()
    mirror_id = result.get("id")

    print(f"Mirrored Database '{mirrored_database_name}' created. ID: {mirror_id}")
    return mirror_id

# --------------------------------------------------------
#  Mirrored Databricks Catalog creation function
# --------------------------------------------------------
def create_mirrored_databricks_catalog(
        access_token: str,
        databricks_mirror_name: str,
        databricks_url: str,
        databricks_catalog_name: str,
        tenant_id: str,
        client_id: str,
        client_secret: str
):
    """
    Create a Fabric Mirrored Azure Databricks Catalog using a Service Principal.
    """

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/mirroredAzureDatabricksCatalogs"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": databricks_mirror_name,
        "source": {
            "workspaceUrl": databricks_url,
            "catalogName": databricks_catalog_name,
            "authentication": {
                "type": "ServicePrincipal",
                "tenantId": tenant_id,
                "clientId": client_id,
                "clientSecret": client_secret
            }
        }
    }

    response = requests.post(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Mirrored Databricks Catalog '{databricks_mirror_name}' created successfully using Service Principal auth.")

## **Mirror Helper functions**

In [ ]:
# --------------------------------------------------------    
#  Validate database connection function
# --------------------------------------------------------

validate_mirror_source_types = {
    "SqlDatabase", # Azure SQL Database
    "SqlManagedInstance", # Azure SQL Managed Instance
    "SqlServer", # On premise SQL Server
    "AzurePostgreSql", # Azure Postgres SQL
    "CosmosDb" # Azure Cosmos DB NoSQL only
}

def validate_mirror_source_type(source_type: str):
    """
    Validates the source_type for a Fabric Mirrored Database.
    Throws a ValueError if someone decides to freestyle.
    """
    if source_type not in validate_mirror_source_types:
        raise ValueError(
            f"Invalid source_type '{source_type}'. "
            f"Valid options: {', '.join(validate_mirror_source_types)}"
        )


# --------------------------------------------------------    
#  Validate database connection function
# --------------------------------------------------------
def validate_sql_connection(connection_string: str):
    """
    Validates that the SQL connection works using the service principal.
    """
    try:
        conn = pyodbc.connect(connection_string, timeout=5)
        cursor = conn.cursor()
        cursor.execute("SELECT TOP 1 name FROM sys.databases")
        row = cursor.fetchone()
        print("Connection validated. Sample result:", row)
        conn.close()
        return True

    except Exception as ex:
        print("Connection validation failed:", ex)
        return False

# --------------------------------------------------------    
#  Poll mirror provisioning status
# --------------------------------------------------------
def wait_for_mirror_ready(access_token: str, mirror_id: str, timeout_seconds=300, poll_interval=5):
    """
    Polls Fabric until the mirrored database finishes provisioning.
    Returns the final state.
    """

    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/mirroredDatabases/{mirror_id}"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        response = requests.get(endpoint, headers=headers)
        response.raise_for_status()
        data = response.json()

        state = data.get("state")
        print(f"Mirror status: {state}")

        # Expected end states
        if state in {"Succeeded", "Ready", "Running"}:
            print("Mirrored database is fully provisioned.")
            return data

        # Possible failure states
        if state in {"Failed", "Error", "Canceled"}:
            raise RuntimeError(f"Mirror provisioning failed with state: {state}")

        time.sleep(poll_interval)

    raise TimeoutError("Timed out waiting for mirrored database to finish provisioning.")


## **Transfer functions**

In [ ]:
# --------------------------------------------------------
#  Get Fabric Item Id by display name and type
# --------------------------------------------------------
def get_item_id(access_token: str, item_name: str, item_type: str = None) -> str:
    """
    Retrieve the ID of a Fabric item by its display name (and optional type).
    """
    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()

    items = response.json().get("value", [])

    for item in items:
        if item["displayName"] == item_name:
            if item_type is None or item["type"] == item_type:
                return item["id"]

    raise ValueError(f"Item '{item_name}' (type={item_type}) not found in workspace.")


# --------------------------------------------------------    
#  Tranfer Fabric Item ownership to Service Principal function
# --------------------------------------------------------
def assign_item_owner(access_token: str, workspace_id: str, item_id: str, principal_id: str):
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{item_id}/permissions"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "grantee": {
            "principalType": "App",
            "principalId": principal_id
        },
        "role": "Owner"
    }

    response = requests.patch(endpoint, headers=headers, json=body)
    response.raise_for_status()

    print(f"Service Principal now owns item {item_id}.")

# --------------------------------------------------------    
#  Test Service Principal permissions function
# --------------------------------------------------------
def test_permissions_api(access_token, workspace_id, item_id):
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{item_id}/permissions"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    r = requests.get(endpoint, headers=headers)
    print("Status:", r.status_code)
    print("Body:", r.text)

# **Operation**

## **Get access token**

In [ ]:
# --------------------------------------------------------    
#  Get access token
# --------------------------------------------------------
access_token = get_access_token(tenant_id, client_id, client_secret)

# --------------------------------------------------------    
#  Get current Workspace Id
# --------------------------------------------------------
workspace_id = spark.conf.get("trident.workspace.id")
print(f"Workspace Id: {workspace_id}")

# --------------------------------------------------------    
#  Get current Capacity Id
# --------------------------------------------------------
capacity_id = get_capacity_id(access_token, workspace_id)
print("Capacity Id:", capacity_id)

### **Transfer item ownership (Currently not supported unless a Power BI artifact)** 

#### **Supported items: Datasets, Reports, Dashboards, Dataflows Gen1, Warehouses**

In [ ]:
# --------------------------------------------------------    
#  Get Fabric Item Id
# --------------------------------------------------------
item_name = "Config_Warehouse"
item_type = "Warehouse"
item_id = get_item_id(access_token, item_name, item_type)
print(f"{item_name}: {item_id}")

# --------------------------------------------------------    
#  Test Service Principal permissions
# --------------------------------------------------------
test_permissions_api(access_token, workspace_id, client_id)

# --------------------------------------------------------    
#  Tranfer ownership of Fabric Item to Service Principal
# --------------------------------------------------------
# assign_item_owner(access_token, workspace_id, item_id, client_id)

### **Transfer warehouse ownership**

In [ ]:
# --------------------------------------------------------    
#  Get Warehouse Id
# --------------------------------------------------------
item_name = "Config_Warehouse"
item_type = "Warehouse"
item_id = get_item_id(access_token, item_name, item_type)
print(f"{item_name}: {item_id}")

warehouse_name = item_name
warehouse_id = item_id

# Get token
app = msal.ConfidentialClientApplication(
    client_id
    , authority=f"https://login.microsoftonline.com/{tenant_id}"
    , client_credential=client_secret
)
token = app.acquire_token_for_client(scopes=["https://analysis.windows.net/powerbi/api/.default"])["access_token"]

# Takeover
url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datawarehouses/{warehouse_id}/takeover"
response = requests.post(url, headers={"Authorization": f"Bearer {token}"})

if response.status_code == 200:
    print(f"{warehouse_name} ownership successfully taken over.")
else:
    print(f"Failed: {response.status_code} - {response.text}")

### **Create single Fabric item**

In [ ]:
# --------------------------------------------------------    
#  Create Workspace with Service Principal
# --------------------------------------------------------
workspace_name = 'WORKSPACE'
create_workspace(access_token,f"{workspace_name}", f"{capacity_id}")

# --------------------------------------------------------    
#  Create Lakehouse with Service Principal
# --------------------------------------------------------
lakehouse_name = 'LAKEHOUSE'
create_lakehouse(access_token, lakehouse_name, True) #True for schema, False for no schema

# --------------------------------------------------------    
#  Create Notebook with Service Principal
# --------------------------------------------------------
notebook_name = 'NOTEBOOK'
create_notebook(access_token, notebook_name)

# --------------------------------------------------------    
#  Create Warehouse with Service Principal
# --------------------------------------------------------
warehouse_name = 'WAREHOUSE'
create_warehouse(access_token, warehouse_name)

# --------------------------------------------------------    
#  Create Pipeline with Service Principal
# --------------------------------------------------------
pipeline_name = 'PIPELINE'
create_pipeline(access_token, pipeline_name)

# --------------------------------------------------------    
#  Create Dataflow with Service Principal
# --------------------------------------------------------
dataflow_name = 'DATAFLOW'
create_dataflow_gen2(access_token, dataflow_name)

# --------------------------------------------------------    
#  Create SQL Server with Service Principal
# --------------------------------------------------------
sql_server_name = 'Config_Database'
create_sql_server(access_token, sql_server_name)

# --------------------------------------------------------    
#  Create Variable Library with Service Principal
# --------------------------------------------------------
variable_library_name = "VARIABLE_LIBRARY"
create_variable_library(access_token, variable_library_name)

# --------------------------------------------------------    
#  Create Eventstream with Service Principal
# --------------------------------------------------------
evenstream_name = 'EVENTSTREAM'
create_eventstream(access_token, evenstream_name)

# --------------------------------------------------------    
#  Create Eventstream with Service Principal
# --------------------------------------------------------
eventhouse_name = 'EVENTHOUSE'
create_eventhouse(access_token, eventhouse_name)

# --------------------------------------------------------    
#  Create Ontology with Service Principal
# --------------------------------------------------------
ontology_name = 'ONTOLOGY'
create_ontology(access_token, ontology_name)

# --------------------------------------------------------    
#  Create Reflex (Activator) with Service Principal
# --------------------------------------------------------
reflex_name = 'REFLEX'
create_reflex(access_token, reflex_name)

# --------------------------------------------------------    
#  Create Mirrored Database with Service Principal
# --------------------------------------------------------
mirrored_database_name = "MIRRORED_DATABASE"
mirrored_database_type = "SqlDatabase"
mirrored_database_conn = "<DatabaseConnectionString>"

# Validate the source type
validate_mirror_source_type(mirrored_database_type)

# Validate connection before provisioning mirrored database
if not validate_sql_connection(mirrored_database_conn):
    print("Mirror not created due to failed SQL connection test.")
else:
    # Create mirror and get Id
    mirror_id = create_mirrored_database(
        access_token,
        mirrored_database_name,
        mirrored_database_type,
        mirrored_database_conn
    )

    # Wait until provisioned
    final_state = wait_for_mirror_ready(access_token, mirror_id)

    print("Final mirror metadata:")
    print(final_state)

# --------------------------------------------------------    
#  Create Databricks Mirror with Service Principal
# --------------------------------------------------------

# Fabric Item display name
databricks_mirror_name = "dna_prod"

# Databricks variables
databricks_url = "https://adb-6002889854638527.7.azuredatabricks.net"
databricks_catalog_name = "tuatara"

create_mirrored_databricks_catalog(
    access_token,
    databricks_mirror_name,
    databricks_url,
    databricks_catalog_name,
    tenant_id,
    client_id,
    client_secret
)

### **Create multiple Fabric items**

In [ ]:
# --------------------------------------------------------    
#  Define creators for multi item deployment
# --------------------------------------------------------
creators = {
    "lakehouse": lambda token, name: create_lakehouse(token, name, True),
    "notebook": create_notebook,
    "warehouse": create_warehouse,
    "pipeline": create_pipeline,
    "dataflow": create_dataflow_gen2,
    "variable_library": create_variable_library
}

# --------------------------------------------------------    
#  Create list of items by type
# --------------------------------------------------------
items_to_create = [
    ("lakehouse", ["Bronze_Data_Lakehouse"]),
    ("notebook", ["Extract_Metadata", "Merge_Operation", "Pipeline_Config"]),
    ("warehouse", ["Config_Warehouse"]),
    ("pipeline", ["Pipeline_01", "Pipeline_02", "Pipeline_03"]),
    ("dataflow", []),
    ("variable_library", ["Config_Metadata", "Config_Source"])
]

# --------------------------------------------------------    
#  Loop through lists and create defined items
# --------------------------------------------------------
for item_type, names in items_to_create:
    for name in names:
        creators[item_type](access_token, name)